# Airline Passenger Demand Forecasting (Time Series)

**Goal:** forecast monthly airline passenger volume from historical totals, the kind of forecast an airline or airport uses for capacity planning, staffing, and revenue management.

**Dataset:** [Airline Passengers](https://github.com/jbrownlee/Datasets) — classic monthly international airline passenger totals, Jan 1949 to Dec 1960 (144 months), in thousands. Downloaded automatically at runtime (see below) — not committed to this repo.

**Approach:**
1. Load data via a small, reproducible download step (with a fallback mirror)
2. Explore trend, seasonality, and stationarity
3. Time-based train/test split (never shuffle time series — the test set is strictly the most recent months)
4. Train and compare two models: **SARIMA** (classical statistical model, captures trend + seasonality explicitly) vs. **LSTM** (deep learning, learns temporal patterns from windowed sequences)
5. Evaluate with RMSE, MAE, and MAPE on the held-out months

**Reproducibility:** fixed random seeds throughout, pinned dependency versions in `requirements.txt`, no absolute file paths, no manual steps required to reproduce results end-to-end.


## 1. Setup

In [ ]:
import os
import io
import time
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)


## 2. Data loading (reproducible, no committed CSV)

The raw dataset (~2 KB, 144 rows) is **not** stored in this repo. Instead, this cell downloads it
directly from its public source at runtime, with a fallback mirror in case the primary source is
unavailable, and caches it locally under `data/` (git-ignored) so repeated runs don't re-download.

This keeps the repo small and means anyone who clones this repo gets the data automatically —
no manual download step.

In [ ]:
DATA_DIR = "data"
LOCAL_PATH = os.path.join(DATA_DIR, "airline-passengers.csv")

DATA_SOURCES = [
    "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv",
    "https://raw.githubusercontent.com/swilsonmfc/pandas/main/AirPassengers.csv",
]

def _normalize_columns(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Different public mirrors of this dataset use slightly different column names
    (e.g. 'Passengers' vs '#Passengers'). Normalize to ['Month', 'Passengers']."""
    cols = {c: c.strip().lstrip("#") for c in raw_df.columns}
    raw_df = raw_df.rename(columns=cols)
    raw_df = raw_df.rename(columns={raw_df.columns[0]: "Month", raw_df.columns[1]: "Passengers"})
    return raw_df[["Month", "Passengers"]]

def load_airline_data(local_path: str = LOCAL_PATH, sources=DATA_SOURCES, timeout: int = 15) -> pd.DataFrame:
    """Load the Airline Passengers dataset, downloading it once and caching it locally.

    Tries each URL in `sources` in order and falls back to the next on failure, so a single
    dead mirror does not break reproducibility for anyone re-running this notebook.
    """
    if os.path.exists(local_path):
        return pd.read_csv(local_path, parse_dates=["Month"], index_col="Month")

    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    last_error = None
    for url in sources:
        try:
            response = requests.get(url, timeout=timeout)
            response.raise_for_status()
            raw_df = pd.read_csv(io.StringIO(response.text))
            df = _normalize_columns(raw_df)
            df.to_csv(local_path, index=False)
            print(f"Downloaded dataset from: {url}")
            return pd.read_csv(local_path, parse_dates=["Month"], index_col="Month")
        except Exception as exc:  # noqa: BLE001 - we want to try every mirror before failing
            last_error = exc
            continue

    raise RuntimeError(
        f"Could not download the dataset from any configured source. Last error: {last_error}"
    )

df = load_airline_data()
df = df.asfreq("MS")  # ensure a clean monthly-start frequency, required by SARIMAX
print(df.shape)
df.head()


## 3. Exploratory analysis: trend, seasonality, stationarity

In [ ]:
df["Passengers"].plot(title="Monthly airline passengers, 1949-1960")
plt.ylabel("Passengers (thousands)")
plt.show()

print("Missing values:", df["Passengers"].isna().sum())


In [ ]:
decomposition = seasonal_decompose(df["Passengers"], model="multiplicative", period=12)
fig = decomposition.plot()
fig.set_size_inches(10, 7)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(df["Passengers"], lags=36, ax=axes[0])
plot_pacf(df["Passengers"], lags=36, ax=axes[1])
plt.tight_layout()
plt.show()


**Observations:**
- Clear upward trend and strong 12-month seasonality (summer peaks), with seasonal swings that *grow* over time — this is a **multiplicative** pattern, which is why the decomposition above uses `model="multiplicative"` and why SARIMA is fit on the log-transformed series below (log-transforming a multiplicative series makes its seasonality additive, which SARIMA assumes).
- The ACF shows slow decay and a repeating spike every 12 lags — textbook evidence of both trend (needs differencing) and yearly seasonality (needs seasonal differencing).

## 4. Train / test split

Time series must **never** be split randomly — the test set has to be the most recent, unseen period, otherwise the model would be evaluated on data it could indirectly "see" via nearby training points. The last 24 months (2 full seasonal cycles) are held out as the test set.

In [ ]:
TEST_MONTHS = 24

train = df.iloc[:-TEST_MONTHS]
test = df.iloc[-TEST_MONTHS:]
print(f"Train: {train.index.min().date()} to {train.index.max().date()} ({len(train)} months)")
print(f"Test:  {test.index.min().date()} to {test.index.max().date()} ({len(test)} months)")


## 5. Model 1 — SARIMA (classical)

Fit on the **log-transformed** series to turn the multiplicative seasonality into an additive one.
Order `(1,1,1)x(1,1,1,12)` is a standard, well-documented starting point for this exact dataset
(non-seasonal + seasonal differencing to handle trend and yearly seasonality, plus one AR and one MA
term at each level).

In [ ]:
train_log = np.log(train["Passengers"])

sarima_model = SARIMAX(
    train_log,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_train_start = time.perf_counter()
sarima_fit = sarima_model.fit(disp=False)
sarima_train_time = time.perf_counter() - sarima_train_start
print(f"SARIMA training time: {sarima_train_time:.3f} sec")
print(sarima_fit.summary().tables[0])


In [ ]:
sarima_predict_start = time.perf_counter()
sarima_forecast_log = sarima_fit.get_forecast(steps=TEST_MONTHS)
sarima_pred = np.exp(sarima_forecast_log.predicted_mean)
sarima_predict_time = time.perf_counter() - sarima_predict_start
print(f"SARIMA prediction time ({TEST_MONTHS} months): {sarima_predict_time:.4f} sec")

sarima_ci = np.exp(sarima_forecast_log.conf_int())

plt.plot(train.index, train["Passengers"], label="Train")
plt.plot(test.index, test["Passengers"], label="Actual (test)")
plt.plot(test.index, sarima_pred, label="SARIMA forecast", linestyle="--")
plt.fill_between(test.index, sarima_ci.iloc[:, 0], sarima_ci.iloc[:, 1], alpha=0.2, label="95% CI")
plt.legend()
plt.title("SARIMA forecast vs actual")
plt.show()


## 6. Model 2 — LSTM (deep learning)

The LSTM needs the data scaled to [0, 1] and reshaped into overlapping windows (`LOOKBACK` months of
history predicting the next month). The scaler is fit **only on the training set** to avoid leaking
test-set information into the transform — the same discipline as the `ColumnTransformer` pipelines
used in the classification/regression projects.

In [ ]:
LOOKBACK = 12

scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train[["Passengers"]])
# Test windows need the LOOKBACK months immediately preceding them, so include the tail of train.
full_scaled = scaler.transform(df[["Passengers"]])

def make_windows(series: np.ndarray, lookback: int):
    X, y = [], []
    for i in range(lookback, len(series)):
        X.append(series[i - lookback:i, 0])
        y.append(series[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = make_windows(train_scaled, LOOKBACK)
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
print("Training windows:", X_train.shape)


In [ ]:
lstm_model = Sequential([
    LSTM(50, activation="tanh", input_shape=(LOOKBACK, 1)),
    Dense(1),
])
lstm_model.compile(optimizer="adam", loss="mse")

lstm_train_start = time.perf_counter()
history = lstm_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=8,
    verbose=0,
    shuffle=False,  # preserve temporal order within batches
)
lstm_train_time = time.perf_counter() - lstm_train_start
print(f"LSTM training time (100 epochs): {lstm_train_time:.3f} sec")

plt.plot(history.history["loss"])
plt.title("LSTM training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE (scaled)")
plt.show()


**Forecasting strategy:** walk-forward one-step-ahead. At each test month, the model predicts the
next value using the *actual* previous `LOOKBACK` months (not its own prior predictions), which is the
standard, honest way to evaluate a one-step forecaster and matches how SARIMA's one-step errors are
implicitly evaluated over the forecast horizon above.

In [ ]:
lstm_predict_start = time.perf_counter()
test_predictions_scaled = []
for i in range(len(test)):
    window = full_scaled[len(train) + i - LOOKBACK: len(train) + i, 0]
    window = window.reshape((1, LOOKBACK, 1))
    pred = lstm_model.predict(window, verbose=0)
    test_predictions_scaled.append(pred[0, 0])
lstm_predict_time = time.perf_counter() - lstm_predict_start
print(f"LSTM prediction time ({TEST_MONTHS} one-step forecasts): {lstm_predict_time:.4f} sec")

lstm_pred = scaler.inverse_transform(np.array(test_predictions_scaled).reshape(-1, 1)).flatten()
lstm_pred = pd.Series(lstm_pred, index=test.index)

plt.plot(train.index, train["Passengers"], label="Train")
plt.plot(test.index, test["Passengers"], label="Actual (test)")
plt.plot(test.index, lstm_pred, label="LSTM forecast", linestyle="--")
plt.legend()
plt.title("LSTM forecast vs actual")
plt.show()


## 7. Evaluation

In [ ]:
def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def evaluate(name, y_true, y_pred):
    return {
        "model": name,
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mae": mean_absolute_error(y_true, y_pred),
        "mape_%": mape(y_true.values, y_pred.values),
    }

results_df = pd.DataFrame([
    evaluate("SARIMA", test["Passengers"], sarima_pred),
    evaluate("LSTM", test["Passengers"], lstm_pred),
]).set_index("model").round(2)

results_df


### Runtime comparison

Predictive accuracy isn't the only thing that matters in practice — training and inference cost affect
what's actually deployable, especially for frequent retraining or many-series forecasting at scale.

In [ ]:
runtime_df = pd.DataFrame([
    {"model": "SARIMA", "train_time_sec": sarima_train_time, "predict_time_sec": sarima_predict_time},
    {"model": "LSTM", "train_time_sec": lstm_train_time, "predict_time_sec": lstm_predict_time},
]).set_index("model").round(4)

runtime_df["predict_time_per_forecast_ms"] = (runtime_df["predict_time_sec"] / TEST_MONTHS) * 1000
runtime_df


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
runtime_df[["train_time_sec"]].plot(kind="bar", ax=ax, legend=False, color="#4C72B0")
ax.set_ylabel("Training time (seconds, log scale)")
ax.set_yscale("log")
ax.set_title("Training time: SARIMA vs LSTM")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
plt.plot(train.index[-24:], train["Passengers"].iloc[-24:], label="Train (last 24mo)", color="grey")
plt.plot(test.index, test["Passengers"], label="Actual", color="black", linewidth=2)
plt.plot(test.index, sarima_pred, label="SARIMA", linestyle="--")
plt.plot(test.index, lstm_pred, label="LSTM", linestyle="--")
plt.legend()
plt.title("Model comparison: SARIMA vs LSTM on held-out test months")
plt.show()


## 8. Conclusion

- **SARIMA vs. LSTM:** compare the `results_df` table above. On a clean, strongly seasonal series like this one with limited data (144 points), SARIMA is a very hard baseline to beat — it directly encodes the trend and 12-month seasonality the data actually has, and doesn't need much data to estimate a handful of parameters. LSTM can match or edge it out with enough data and tuning, but on a short series like this it's more prone to overfitting or underfitting depending on the epoch count and lookback window, which is itself a useful, honest finding to report rather than a limitation to hide.
- **Runtime vs. accuracy trade-off:** see the `runtime_df` table above — SARIMA's training is a closed-form-ish maximum likelihood fit over a handful of parameters, while the LSTM runs 100 epochs of gradient descent, so expect an order-of-magnitude (or more) gap in training time in SARIMA's favor on a dataset this size. That gap matters directly if this pipeline needs to retrain frequently or scale to many series at once.
- **Practical takeaway:** for short, clearly-structured seasonal series, a well-specified classical model is often the pragmatic choice on both accuracy *and* cost grounds — reach for deep learning when there's substantially more data, multiple related series, or exogenous variables SARIMA can't easily incorporate, where its extra training cost is easier to justify.
- **Reproducibility:** re-running this notebook top to bottom reproduces the same split, same SARIMA fit, and same LSTM training run, because random seeds are fixed for both NumPy and TensorFlow, and the data is pulled from the same versioned source each time.

**Possible extensions:** grid search over SARIMA orders (`pmdarima.auto_arima`), a stacked or bidirectional LSTM, adding exogenous regressors (e.g. holidays), or expanding to a multi-series dataset where deep learning has a clearer data advantage.
